In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from collections import Counter
import math

In [3]:
df = pd.read_parquet("20240609.parquet")

print(df.shape)
df.head()

(311084, 37)


,DST_IP,DST_IP_SUBNET,DST_IP_VERSION,DST_ASN,DST_COUNTRY,DST_PORT,PROTOCOL,TIME_FIRST,TIME_LAST,DURATION,...,QUIC_TLS_EXT_TYPE,QUIC_PACKETS,PPI,PPI_LEN,PPI_DURATION,PPI_ROUNDTRIPS,PHIST_SRC_SIZES,PHIST_DST_SIZES,PHIST_SRC_IPT,PHIST_DST_IPT
0,775437f803,ae12b508c7,4,15169,US,443,17,2024-06-08 23:00:00+02:00,2024-06-08 23:00:20.936360+02:00,20.936360,...,"[0, 23, 65281, 10, 16, 5, 34, 51, 42, 43, 13, ...","[131, 130, 133, 128, 128, 128, 128, 128, 128, ...","[[0, 0, 15, 0, 0, 0, 0, 0, 0, 0, 1, 0, 11, 0, ...",30,0.121,6,"[0, 1, 12, 1, 0, 8, 0, 1]","[0, 8, 0, 2, 0, 3, 4, 16]","[12, 7, 1, 0, 0, 0, 0, 2]","[22, 3, 5, 0, 0, 0, 0, 2]"
1,e9f73131b9,a05083b596,4,15169,US,443,17,2024-06-08 23:00:00+02:00,2024-06-08 23:00:26.380968+02:00,26.380968,...,"[0, 10, 16, 13, 51, 45, 43, 57]","[129, 133, 132, 132, 133, 132, 132, 132, 129, ...","[[0, 2, 0, 0, 3, 0, 0, 0, 3, 2, 6, 0, 2, 5, 11...",30,0.111,9,"[0, 14, 22, 0, 1, 4, 10, 22]","[0, 28, 0, 7, 5, 22, 3, 17]","[49, 8, 4, 5, 0, 0, 1, 5]","[48, 11, 8, 5, 1, 2, 0, 6]"
2,f2ac2651bf,ae12b508c7,4,15169,US,443,17,2024-06-08 23:00:00+02:00,2024-06-08 23:00:00.097876+02:00,0.097876,...,"[65037, 17513, 45, 51, 10, 42, 57, 27, 13, 0, ...","[129, 130, 133, 128, 128, 133, 132, 128, 128, ...","[[0, 0, 15, 0, 0, 5, 0, 1, 1, 0, 1, 6, 0, 0, 0...",23,0.098,5,"[0, 1, 3, 2, 0, 1, 0, 6]","[0, 4, 0, 1, 2, 1, 1, 1]","[9, 2, 1, 0, 0, 0, 0, 0]","[7, 1, 1, 0, 0, 0, 0, 0]"
3,2d44d5c4f3,ae12b508c7,4,15169,US,443,17,2024-06-08 23:00:00+02:00,2024-06-08 23:00:00.396463+02:00,0.396463,...,"[16, 17513, 51, 45, 0, 57, 10, 27, 65037, 43, 13]","[129, 129, 132, 132, 129, 132, 132, 132, 132, ...","[[0, 1, 14, 0, 134, 0, 0, 0, 2, 12, 147, 1, 0,...",22,0.396,5,"[0, 0, 7, 2, 1, 0, 0, 2]","[0, 2, 0, 1, 1, 1, 1, 4]","[8, 0, 1, 0, 2, 0, 0, 0]","[6, 0, 1, 0, 2, 0, 0, 0]"
4,bcd7b62c53,34635d709c,4,15169,US,443,17,2024-06-08 23:00:00+02:00,2024-06-08 23:00:00.291261+02:00,0.291261,...,"[43690, 0, 10, 16, 5, 13, 18, 51, 45, 43, 57, ...","[129, 129, 132, 132, 132, 129, 132, 132, 132, ...","[[0, 14, 0, 0, 263, 5, 0, 9, 0, 0, 0], [1, -1,...",11,0.291,2,"[0, 0, 2, 0, 0, 0, 0, 2]","[0, 0, 0, 0, 0, 0, 0, 7]","[2, 0, 0, 0, 0, 1, 0, 0]","[5, 0, 0, 0, 0, 1, 0, 0]"


In [4]:
ppi_features = df[
    [
        "PPI",
        "PPI_LEN",
        "PPI_DURATION",
        "PPI_ROUNDTRIPS"
    ]
].copy()

ppi_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 311084 entries, 0 to 311083
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PPI             311084 non-null  object 
 1   PPI_LEN         311084 non-null  int64  
 2   PPI_DURATION    311084 non-null  float64
 3   PPI_ROUNDTRIPS  311084 non-null  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 9.5+ MB


In [5]:
# Calculate the average inter-packet time for each flow
ppi_features["mean_ipt"] = ppi_features["PPI"].apply(
    lambda x: np.mean(x[0])
)

In [6]:
# Calculate the variability of inter-packet times
ppi_features["std_ipt"] = ppi_features["PPI"].apply(
    lambda x: np.std(x[0])
)

In [7]:
# Calculate the average packet size
ppi_features["mean_packet_size"] = ppi_features["PPI"].apply(
    lambda x: np.mean(x[2])
)

In [8]:
# Calculate packet size variability
ppi_features["std_packet_size"] = ppi_features["PPI"].apply(
    lambda x: np.std(x[2])
)

In [9]:
def direction_change_ratio(directions):

    # If there is only one packet, no direction change is possible
    if len(directions) < 2:
        return 0

    changes = 0

    # Compare every packet direction with the next one
    for i in range(len(directions) - 1):

        if directions[i] != directions[i + 1]:
            changes += 1

    # Return the proportion of direction changes
    return changes / (len(directions) - 1)

In [10]:
ppi_features["direction_change_ratio"] = (
    ppi_features["PPI"].apply(
        lambda x: direction_change_ratio(x[1])
    )
)

In [11]:
def forward_packet_ratio(directions):

    if len(directions) == 0:
        return 0

    forward_packets = np.sum(directions == 1)

    return forward_packets / len(directions)

In [12]:
ppi_features["forward_packet_ratio"] = (
    ppi_features["PPI"].apply(
        lambda x: forward_packet_ratio(x[1])
    )
)

In [13]:
ppi_features["log_ppi_duration"] = np.log1p(ppi_features["PPI_DURATION"])
ppi_features["log_mean_ipt"] = np.log1p(ppi_features["mean_ipt"])
ppi_features["log_std_ipt"] = np.log1p(ppi_features["std_ipt"])

In [14]:
ppi_final = ppi_features[
    [
        # Original Features
        "PPI_LEN",
        "PPI_ROUNDTRIPS",

        # Log-transformed Features
        "log_ppi_duration",
        "log_mean_ipt",
        "log_std_ipt",

        # Engineered Features
        "mean_packet_size",
        "std_packet_size",
        "direction_change_ratio",
        "forward_packet_ratio"
    ]
].copy()

print("Final PPI Feature Bank Shape:", ppi_final.shape)

ppi_final.head()

Final PPI Feature Bank Shape: (311084, 9)


,PPI_LEN,PPI_ROUNDTRIPS,log_ppi_duration,log_mean_ipt,log_std_ipt,mean_packet_size,std_packet_size,direction_change_ratio,forward_packet_ratio
0,30,6,0.114221,1.616082,2.182880,614.100000,576.196341,0.413793,0.366667
1,30,9,0.105261,1.547563,1.857983,545.433333,512.401189,0.586207,0.433333
2,23,5,0.093490,1.660296,2.318133,478.391304,526.801236,0.454545,0.565217
3,22,5,0.333611,2.944439,3.708362,447.181818,530.688715,0.476190,0.545455
4,11,2,0.255417,3.312532,4.329917,982.272727,445.369324,0.300000,0.363636


In [15]:
cid_features = df[
    [
        "DST_ASN",
        "QUIC_OCCID",
        "QUIC_OSCID",
        "QUIC_SCID",
        "QUIC_RETRY_SCID",
        "QUIC_SNI"
    ]
].copy()

cid_features.head()

,DST_ASN,QUIC_OCCID,QUIC_OSCID,QUIC_SCID,QUIC_RETRY_SCID,QUIC_SNI
0,15169,c722b0,a0276a2b8010cde6ca,e0276a2b8010cde6,,yt3.ggpht.com
1,15169,12fb15cb4b52e601773a1cc3624c7ced,ecad66527b4eb8b0171b1a817c3c3e21,ecad66527b4eb8b0,,gew4-spclient.spotify.com
2,15169,,a44ff8ada0a09e1f,e44ff8ada0a09e1f,,play.googleapis.com
3,15169,,19e85983c29be24f,f9e85983c29be24f,,i.ytimg.com
4,15169,,16215deaba93d58f,f6215deaba93d58f,,s.youtube.com


In [16]:
cid_features["occid_length"] = cid_features["QUIC_OCCID"].str.len()

cid_features["oscid_length"] = cid_features["QUIC_OSCID"].str.len()

cid_features["scid_length"] = cid_features["QUIC_SCID"].str.len()

cid_features["retry_length"] = cid_features["QUIC_RETRY_SCID"].str.len()

cid_features["sni_length"] = cid_features["QUIC_SNI"].str.len()

cid_features["sni_labels"] = (
    cid_features["QUIC_SNI"]
    .str.split(".")
    .str.len()
)

In [17]:
cid_features["retry_present"] = (cid_features["retry_length"] > 0).astype(int)

In [18]:
def shannon_entropy(text):

    if len(text) == 0:
        return 0.0

    counts = Counter(text)

    entropy = 0.0

    length = len(text)

    for count in counts.values():

        p = count / length

        entropy -= p * math.log2(p)

    return entropy

In [19]:
def normalized_entropy(text):

    if len(text) == 0:
        return 0.0

    H = shannon_entropy(text)

    Hmax = math.log2(min(len(text), 16))

    return H / Hmax if Hmax > 0 else 0.0

In [20]:
cid_features["occid_entropy"] = (cid_features["QUIC_OCCID"].apply(normalized_entropy))

cid_features["oscid_entropy"] = (cid_features["QUIC_OSCID"].apply(normalized_entropy))

cid_features["scid_entropy"] = (cid_features["QUIC_SCID"].apply(normalized_entropy))

In [21]:
cid_features[
    [
        "occid_entropy",
        "oscid_entropy",
        "scid_entropy"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
occid_entropy,311084.0,0.130578,0.316416,0.000000,0.000000,0.000000,0.00000,1.000000
oscid_entropy,311084.0,0.809679,0.056496,0.519455,0.769455,0.812500,0.84375,0.988205
scid_entropy,311084.0,0.803971,0.064494,0.000000,0.769455,0.800705,0.84375,1.000000


In [22]:
MIN_FLOWS = max(50, int(0.001 * len(cid_features)))

In [23]:
asn_groups = cid_features.groupby("DST_ASN")

In [24]:
asn_profile = asn_groups.agg({

    "scid_length": ["count", "median"],

    "scid_entropy": "median",

    "oscid_length": "median",

    "oscid_entropy": "median",

    "retry_present": "mean",

    "sni_length": "median",

    "sni_labels": "median"

})

In [25]:
asn_profile.columns = [

    "flows",
    "median_scid_length",
    "median_scid_entropy",
    "median_oscid_length",
    "median_oscid_entropy",
    "retry_rate",
    "median_sni_length",
    "median_sni_labels",

]

In [26]:
asn_profile = asn_profile[asn_profile["flows"] >= MIN_FLOWS]

In [27]:
asn_profile.head()

,flows,median_scid_length,median_scid_entropy,median_oscid_length,median_oscid_entropy,retry_rate,median_sni_length,median_sni_labels
DST_ASN,,,,,,,,
6185,655,40.0,0.925925,16.0,0.800705,0.000000,17.0,3.0
8075,2809,28.0,0.884787,16.0,0.800705,0.000356,18.0,3.0
13335,7832,40.0,0.905643,16.0,0.800705,0.000000,14.0,3.0
15169,238916,16.0,0.800705,16.0,0.812500,0.000000,20.0,3.0
16509,1166,40.0,0.922144,16.0,0.812500,0.000000,18.0,3.0


In [28]:
cid_features = cid_features.merge(
    asn_profile,
    on="DST_ASN",
    how="left"
)

In [29]:
cid_features["known_asn"] = (
    cid_features["flows"].notna().astype(int)
)

In [30]:
global_scid_length = cid_features["scid_length"].median()

global_scid_entropy = cid_features["scid_entropy"].median()

global_oscid_length = cid_features["oscid_length"].median()

global_oscid_entropy = cid_features["oscid_entropy"].median()

global_retry_rate = cid_features["retry_present"].mean()

global_sni_length = cid_features["sni_length"].median()

global_sni_labels = cid_features["sni_labels"].median()

In [31]:
fill_values = {
    "median_scid_length": global_scid_length,
    "median_scid_entropy": global_scid_entropy,
    "median_oscid_length": global_oscid_length,
    "median_oscid_entropy": global_oscid_entropy,
    "retry_rate": global_retry_rate,
    "median_sni_length": global_sni_length,
    "median_sni_labels": global_sni_labels
}

cid_features = cid_features.fillna(fill_values)

In [32]:
asn_profile.describe().T

,count,mean,std,min,25%,50%,75%,max
flows,12.0,25851.916667,67731.826486,411.000000,1056.750000,2397.000000,11168.00000,238916.000000
median_scid_length,12.0,28.500000,11.571910,16.000000,16.000000,31.000000,40.00000,40.000000
median_scid_entropy,12.0,0.863180,0.064243,0.769455,0.800705,0.895215,0.92191,0.925925
median_oscid_length,12.0,16.000000,0.000000,16.000000,16.000000,16.000000,16.00000,16.000000
median_oscid_entropy,12.0,0.807241,0.007145,0.800705,0.800705,0.806602,0.81250,0.820160
retry_rate,12.0,0.000038,0.000104,0.000000,0.000000,0.000000,0.00000,0.000356
median_sni_length,12.0,20.166667,5.890413,14.000000,16.500000,18.000000,21.25000,33.000000
median_sni_labels,12.0,3.083333,0.288675,3.000000,3.000000,3.000000,3.00000,4.000000


In [33]:
asn_profile.sort_values("flows", ascending=False).head(20)

,flows,median_scid_length,median_scid_entropy,median_oscid_length,median_oscid_entropy,retry_rate,median_sni_length,median_sni_labels
DST_ASN,,,,,,,,
15169,238916,16.0,0.800705,16.0,0.812500,0.000000,20.0,3.0
32934,29375,16.0,0.769455,16.0,0.800705,0.000000,18.0,3.0
396982,21176,16.0,0.800705,16.0,0.800705,0.000094,29.0,3.0
13335,7832,40.0,0.905643,16.0,0.800705,0.000000,14.0,3.0
54113,3651,34.0,0.911428,16.0,0.812500,0.000000,20.0,4.0
8075,2809,28.0,0.884787,16.0,0.800705,0.000356,18.0,3.0
20940,1985,16.0,0.788910,16.0,0.800705,0.000000,25.0,3.0
36183,1518,40.0,0.921832,16.0,0.812500,0.000000,15.0,3.0
16509,1166,40.0,0.922144,16.0,0.812500,0.000000,18.0,3.0


In [34]:
cid_features["scid_length_deviation"] = (
    cid_features["scid_length"] -
    cid_features["median_scid_length"]
).abs()

cid_features["oscid_length_deviation"] = (
    cid_features["oscid_length"] -
    cid_features["median_oscid_length"]
).abs()

In [35]:
cid_features["sni_length_deviation"] = (
    cid_features["sni_length"] -
    cid_features["median_sni_length"]
).abs()

In [36]:
cid_features["scid_entropy_deviation"] = (
    cid_features["scid_entropy"] -
    cid_features["median_scid_entropy"]
).abs()

cid_features["oscid_entropy_deviation"] = (
    cid_features["oscid_entropy"] -
    cid_features["median_oscid_entropy"]
).abs()

In [37]:
cid_final = cid_features[
    [
        # Raw CID Features
        "occid_length",
        "oscid_length",
        "scid_length",
        "retry_present",
        "sni_length",
        "sni_labels",

        # Statistical Features
        "oscid_entropy",
        "scid_entropy",

        # Context Features
        "scid_length_deviation",
        "oscid_length_deviation",
        "scid_entropy_deviation",
        "oscid_entropy_deviation",
        "sni_length_deviation",
        "known_asn"
    ]
].copy()


print(cid_final.shape)

(311084, 14)


In [38]:
cid_final.head()

,occid_length,oscid_length,scid_length,retry_present,sni_length,sni_labels,oscid_entropy,scid_entropy,scid_length_deviation,oscid_length_deviation,scid_entropy_deviation,oscid_entropy_deviation,sni_length_deviation,known_asn
0,6,18,16,0,13,3,0.827068,0.831955,0.0,2.0,0.031250,0.014568,7.0,1
1,32,32,16,0,25,3,0.907232,0.863205,0.0,16.0,0.062500,0.094732,5.0,1
2,0,16,16,0,19,3,0.738205,0.757660,0.0,0.0,0.043045,0.074295,1.0,1
3,0,16,16,0,11,3,0.831955,0.800705,0.0,0.0,0.000000,0.019455,9.0,1
4,0,16,16,0,13,3,0.875000,0.875000,0.0,0.0,0.074295,0.062500,7.0,1


In [39]:
hist_features = df[
    [
        "PHIST_SRC_SIZES",
        "PHIST_DST_SIZES",
        "PHIST_SRC_IPT",
        "PHIST_DST_IPT"
    ]
].copy()

In [40]:
def histogram_entropy(hist):

    # Convert to NumPy array
    hist = np.array(hist, dtype=float)

    # Total observations
    total = hist.sum()

    # Handle empty histograms
    if total == 0:
        return 0

    # Convert counts to probabilities
    probabilities = hist / total

    # Calculate Shannon entropy
    entropy = 0

    for p in probabilities:
        if p > 0:
            entropy -= p * math.log2(p)

    # Normalize entropy
    max_entropy = math.log2(len(hist))

    return entropy / max_entropy

In [41]:
# Source packet size histogram entropy
hist_features["src_size_entropy"] = (
    hist_features["PHIST_SRC_SIZES"]
    .apply(histogram_entropy)
)

# Destination packet size histogram entropy
hist_features["dst_size_entropy"] = (
    hist_features["PHIST_DST_SIZES"]
    .apply(histogram_entropy)
)

# Source IPT histogram entropy
hist_features["src_ipt_entropy"] = (
    hist_features["PHIST_SRC_IPT"]
    .apply(histogram_entropy)
)

# Destination IPT histogram entropy
hist_features["dst_ipt_entropy"] = (
    hist_features["PHIST_DST_IPT"]
    .apply(histogram_entropy)
)

In [42]:
hist_features[
    [
        "src_size_entropy",
        "dst_size_entropy",
        "src_ipt_entropy",
        "dst_ipt_entropy"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
src_size_entropy,311084.0,0.588839,0.165378,0.0,0.518203,0.625000,0.705433,0.917811
dst_size_entropy,311084.0,0.596746,0.194245,0.0,0.552581,0.653655,0.716466,0.935785
src_ipt_entropy,311084.0,0.347339,0.180692,0.0,0.231602,0.332962,0.468546,0.982568
dst_ipt_entropy,311084.0,0.338801,0.203510,0.0,0.195320,0.328809,0.482870,0.983404


In [43]:
hist_final = hist_features[
    [
        "src_size_entropy",
        "dst_size_entropy",
        "src_ipt_entropy",
        "dst_ipt_entropy"
    ]
].copy()

print(hist_final.shape)

hist_final.head()

(311084, 4)


,src_size_entropy,dst_size_entropy,src_ipt_entropy,dst_ipt_entropy
0,0.536556,0.643537,0.506613,0.453416
1,0.735708,0.744321,0.527290,0.634883
2,0.662591,0.773976,0.346951,0.328809
3,0.538002,0.773976,0.365265,0.408131
4,0.333333,0.000000,0.306099,0.216674


In [44]:
flow_df = df.copy()

In [45]:
flow_df["duration_safe"] = flow_df["DURATION"].clip(lower=1e-6)

In [46]:
flow_df["total_bytes"] = (
    flow_df["BYTES"] +
    flow_df["BYTES_REV"]
)

In [47]:
flow_df["total_packets"] = (
    flow_df["PACKETS"] +
    flow_df["PACKETS_REV"]
)

In [48]:
flow_df["byte_rate"] = (
    flow_df["total_bytes"] /
    flow_df["duration_safe"]
)

In [49]:
flow_df["packet_rate"] = (
    flow_df["total_packets"] /
    flow_df["duration_safe"]
)

In [50]:
flow_df["avg_packet_size"] = (
    flow_df["total_bytes"] /
    flow_df["total_packets"].clip(lower=1)
)

In [51]:
flow_df["byte_ratio"] = (
    flow_df["BYTES"] /
    flow_df["BYTES_REV"].clip(lower=1)
)

In [52]:
flow_df["packet_ratio"] = (
    flow_df["PACKETS"] /
    flow_df["PACKETS_REV"].clip(lower=1)
)

In [53]:
flow_features = flow_df[
    [
        "DURATION",
        "FLOW_END_REASON",
        "total_bytes",
        "total_packets",
        "byte_rate",
        "packet_rate",
        "avg_packet_size",
        "byte_ratio",
        "packet_ratio"
    ]
]

In [54]:
flow_features.replace([np.inf, -np.inf], np.nan, inplace=True)

flow_features.isnull().sum()

DURATION           0
FLOW_END_REASON    0
total_bytes        0
total_packets      0
byte_rate          0
packet_rate        0
avg_packet_size    0
byte_ratio         0
packet_ratio       0
dtype: int64

In [55]:
flow_features = flow_df[
    [
        "DURATION",
        "FLOW_END_REASON",
        "total_bytes",
        "total_packets",
        "byte_rate",
        "packet_rate",
        "avg_packet_size",
        "byte_ratio",
        "packet_ratio"
    ]
].copy()

In [56]:
flow_features["log_duration"] = np.log1p(flow_features["DURATION"])

flow_features["log_total_bytes"] = np.log1p(flow_features["total_bytes"])

flow_features["log_total_packets"] = np.log1p(flow_features["total_packets"])

flow_features["log_byte_rate"] = np.log1p(flow_features["byte_rate"])

flow_features["log_packet_rate"] = np.log1p(flow_features["packet_rate"])

flow_features["log_byte_ratio"] = np.log1p(flow_features["byte_ratio"])

flow_features["log_packet_ratio"] = np.log1p(flow_features["packet_ratio"])

In [57]:
log_columns = [

"log_duration",

"FLOW_END_REASON",

"log_total_bytes",

"log_total_packets",

"log_byte_rate",

"log_packet_rate",

"avg_packet_size",

"log_byte_ratio",

"log_packet_ratio"

]

flow_features[log_columns].corr()

,log_duration,FLOW_END_REASON,log_total_bytes,log_total_packets,log_byte_rate,log_packet_rate,avg_packet_size,log_byte_ratio,log_packet_ratio
log_duration,1.000000,0.150754,0.521546,0.590588,-0.776654,-0.811062,0.067849,-0.007728,-0.179462
FLOW_END_REASON,0.150754,1.000000,0.094226,0.108655,-0.076448,-0.073317,0.010662,-0.007974,-0.026126
log_total_bytes,0.521546,0.094226,1.000000,0.970622,0.013234,-0.093458,0.582847,-0.177405,-0.428487
log_total_packets,0.590588,0.108655,0.970622,1.000000,-0.099315,-0.170532,0.385169,-0.143204,-0.363948
log_byte_rate,-0.776654,-0.076448,0.013234,-0.099315,1.000000,0.980486,0.376887,-0.094577,-0.100729
log_packet_rate,-0.811062,-0.073317,-0.093458,-0.170532,0.980486,1.000000,0.214877,-0.064283,-0.027210
avg_packet_size,0.067849,0.010662,0.582847,0.385169,0.376887,0.214877,1.000000,-0.204853,-0.481580
log_byte_ratio,-0.007728,-0.007974,-0.177405,-0.143204,-0.094577,-0.064283,-0.204853,1.000000,0.685312
log_packet_ratio,-0.179462,-0.026126,-0.428487,-0.363948,-0.100729,-0.027210,-0.481580,0.685312,1.000000


In [58]:
selected_columns = [

    "log_duration",
    "FLOW_END_REASON",
    "log_total_bytes",
    "log_byte_rate",
    "avg_packet_size",
    "log_byte_ratio",
    "log_packet_ratio"

]

flow_features_selected = flow_features[selected_columns].copy()

flow_features_selected.head()

,log_duration,FLOW_END_REASON,log_total_bytes,log_byte_rate,avg_packet_size,log_byte_ratio,log_packet_ratio
0,3.088146,1,10.349839,7.308989,558.053571,0.164115,0.528844
1,3.309848,1,11.286828,8.014503,514.600000,0.766629,0.636706
2,0.093377,1,9.362890,11.686866,506.391304,1.306060,0.832909
3,0.333943,1,9.254836,10.179950,475.181818,0.396514,0.788457
4,0.255619,1,9.315961,10.549433,1010.272727,0.265230,0.451985


In [59]:
quic_features = df[
    [
        "QUIC_VERSION",
        "QUIC_CLIENT_VERSION",
        "QUIC_TOKEN_LENGTH",
        "QUIC_ZERO_RTT",
        "QUIC_MULTIPLEXED"
    ]
].copy()

quic_features.head()

,QUIC_VERSION,QUIC_CLIENT_VERSION,QUIC_TOKEN_LENGTH,QUIC_ZERO_RTT,QUIC_MULTIPLEXED
0,1,1,70,2,0
1,1,1,70,0,2
2,1,1,70,1,0
3,1,1,0,0,0
4,1,1,0,0,0


In [60]:
quic_features["token_present"] = (
    quic_features["QUIC_TOKEN_LENGTH"] > 0
).astype(int)

In [61]:
quic_features["log_token_length"] = np.log1p(
    quic_features["QUIC_TOKEN_LENGTH"]
)

In [62]:
quic_features["zero_rtt_present"] = (
    quic_features["QUIC_ZERO_RTT"] > 0
).astype(int)

In [63]:
quic_features["multiplexed"] = (
    quic_features["QUIC_MULTIPLEXED"] > 0
).astype(int)

In [64]:
version_map = {
    version: idx
    for idx, version in enumerate(
        sorted(quic_features["QUIC_VERSION"].unique())
    )
}

quic_features["quic_version_id"] = (
    quic_features["QUIC_VERSION"].map(version_map)
)

In [65]:
version_map

{np.int64(0): 0,
 np.int64(1): 1,
 np.int64(3467641594): 2,
 np.int64(4207849474): 3,
 np.int64(4207849486): 4,
 np.int64(4207849491): 5,
 np.int64(4278190109): 6}

In [66]:
quic_features["log_zero_rtt"] = np.log1p(
    quic_features["QUIC_ZERO_RTT"]
)

In [67]:
quic_features["log_multiplexed"] = np.log1p(
    quic_features["QUIC_MULTIPLEXED"]
)

In [68]:
quic_selected = quic_features[
    [
        "quic_version_id",
        "log_token_length",
        "log_zero_rtt",
        "zero_rtt_present",
        "log_multiplexed",
        "multiplexed"
    ]
]

quic_selected.corr()

,quic_version_id,log_token_length,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
quic_version_id,1.000000,0.132754,-0.110712,-0.126274,-0.043214,-0.044786
log_token_length,0.132754,1.000000,0.642646,0.723075,0.008209,0.007032
log_zero_rtt,-0.110712,0.642646,1.000000,0.886014,-0.047694,-0.066744
zero_rtt_present,-0.126274,0.723075,0.886014,1.000000,-0.079850,-0.092396
log_multiplexed,-0.043214,0.008209,-0.047694,-0.079850,1.000000,0.960850
multiplexed,-0.044786,0.007032,-0.066744,-0.092396,0.960850,1.000000


In [69]:
selected_quic = quic_features[
    [
        "quic_version_id",
        "QUIC_TOKEN_LENGTH",
        "log_zero_rtt",
        "zero_rtt_present",
        "log_multiplexed",
        "multiplexed"
    ]
].copy()

selected_quic.head()

,quic_version_id,QUIC_TOKEN_LENGTH,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
0,1,70,1.098612,1,0.000000,0
1,1,70,0.000000,0,1.098612,1
2,1,70,0.693147,1,0.000000,0
3,1,0,0.000000,0,0.000000,0
4,1,0,0.000000,0,0.000000,0


In [70]:
with_CID_df = pd.concat([ppi_final,cid_final,hist_final,flow_features_selected,selected_quic],axis=1)

print(with_CID_df.shape)

with_CID_df.to_parquet(
    "with_CID.parquet",
    index=False
)

(311084, 40)


In [71]:
check_df = pd.read_parquet("with_CID.parquet")

print(check_df.shape)
check_df.head()

(311084, 40)


,PPI_LEN,PPI_ROUNDTRIPS,log_ppi_duration,log_mean_ipt,log_std_ipt,mean_packet_size,std_packet_size,direction_change_ratio,forward_packet_ratio,occid_length,...,log_byte_rate,avg_packet_size,log_byte_ratio,log_packet_ratio,quic_version_id,QUIC_TOKEN_LENGTH,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
0,30,6,0.114221,1.616082,2.182880,614.100000,576.196341,0.413793,0.366667,6,...,7.308989,558.053571,0.164115,0.528844,1,70,1.098612,1,0.000000,0
1,30,9,0.105261,1.547563,1.857983,545.433333,512.401189,0.586207,0.433333,32,...,8.014503,514.600000,0.766629,0.636706,1,70,0.000000,0,1.098612,1
2,23,5,0.093490,1.660296,2.318133,478.391304,526.801236,0.454545,0.565217,0,...,11.686866,506.391304,1.306060,0.832909,1,70,0.693147,1,0.000000,0
3,22,5,0.333611,2.944439,3.708362,447.181818,530.688715,0.476190,0.545455,0,...,10.179950,475.181818,0.396514,0.788457,1,0,0.000000,0,0.000000,0
4,11,2,0.255417,3.312532,4.329917,982.272727,445.369324,0.300000,0.363636,0,...,10.549433,1010.272727,0.265230,0.451985,1,0,0.000000,0,0.000000,0


In [72]:
without_CID_df = pd.concat([ppi_final,hist_final,flow_features_selected,selected_quic],axis=1)

print(without_CID_df.shape)

without_CID_df.to_parquet(
    "without_CID.parquet",
    index=False
)

(311084, 26)


In [73]:
check_df2 = pd.read_parquet("without_CID.parquet")

print(check_df2.shape)
check_df2.head()

(311084, 26)


,PPI_LEN,PPI_ROUNDTRIPS,log_ppi_duration,log_mean_ipt,log_std_ipt,mean_packet_size,std_packet_size,direction_change_ratio,forward_packet_ratio,src_size_entropy,...,log_byte_rate,avg_packet_size,log_byte_ratio,log_packet_ratio,quic_version_id,QUIC_TOKEN_LENGTH,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
0,30,6,0.114221,1.616082,2.182880,614.100000,576.196341,0.413793,0.366667,0.536556,...,7.308989,558.053571,0.164115,0.528844,1,70,1.098612,1,0.000000,0
1,30,9,0.105261,1.547563,1.857983,545.433333,512.401189,0.586207,0.433333,0.735708,...,8.014503,514.600000,0.766629,0.636706,1,70,0.000000,0,1.098612,1
2,23,5,0.093490,1.660296,2.318133,478.391304,526.801236,0.454545,0.565217,0.662591,...,11.686866,506.391304,1.306060,0.832909,1,70,0.693147,1,0.000000,0
3,22,5,0.333611,2.944439,3.708362,447.181818,530.688715,0.476190,0.545455,0.538002,...,10.179950,475.181818,0.396514,0.788457,1,0,0.000000,0,0.000000,0
4,11,2,0.255417,3.312532,4.329917,982.272727,445.369324,0.300000,0.363636,0.333333,...,10.549433,1010.272727,0.265230,0.451985,1,0,0.000000,0,0.000000,0


In [74]:
print(with_CID_df.columns.duplicated().sum())
print(without_CID_df.columns.duplicated().sum())

0
0
